# AXI test 4: does it still work on a bigger model? (Qwen2.5-14B)
Model: **Qwen2.5-14B-Instruct in 4-bit**, twice the size of the 7B used in tests 1 to 3. The tasks and the episode scripts are the same as in test 3, so the two models can be compared request by request. Two parts, each with the rule stated in the prompt and without it:
* **Part 1, one step at a time** (about 6 min without the rule, 11 min with it): the same 140 requests of 7 kinds as test 3. Which policy ends with the right graph: *blind* (apply what the model wrote), *repair_m6*, *scope_m7*.
* **Part 2, 30 requests in a row** (about 25 min with the rule, 10 min without it): does the held state stay right? *blind* against *repair_m6* against *scope_m7*.

**Before you start:** Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save. Run each cell **one at a time, top to bottom**, wait for the green tick before the next one. The model download is about 30 GB (about 10 minutes); the whole thing takes about 70 minutes.

If Colab disconnects: reconnect, run cells 1 to 4 again, then run the cell you were on again. It continues where it stopped.

The prediction and the reading rules were written before this run in `docs/SCALE_TEST_14B.md`.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-14b.zip`

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the 4-bit tools (about 1 minute)

In [ ]:
!pip -q install bitsandbytes accelerate

### Cell 4: load the model (downloads about 30 GB once; about 10 to 15 minutes). If it stops with a red error about memory or disk, send me the red text

In [ ]:
import sys, json, time, os, shutil, torch
sys.path.insert(0, "/content/axi/python")
print("free disk (GB):", round(shutil.disk_usage("/content").free / 1e9, 0), "(needs about 35)")
from colab.colab_llm_experiment import build_hf_llm
from axi.experiments.long_horizon import run_long, summarize_long, print_long
from axi.experiments.scope_tasks import make_scope_tasks
from axi.experiments.scope_test import run_scope_regime, summarize_scope, print_scope, print_exact, print_verdict
from axi.experiments.bigger_model import verdict_big_single, verdict_big_long, smoke_check, mean_seconds

MODEL = "Qwen/Qwen2.5-14B-Instruct"
llm, model = build_hf_llm(MODEL, "cuda", load_4bit=True)
os.makedirs("/content/out", exist_ok=True)
T0 = time.time()
def prog(i, n): print(f"  {i}/{n}   {time.time() - T0:6.0f} s", end="\r")
def save(res, path):
    json.dump(res, open(path, "w"), indent=1); return path
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))

### Cell 5: quick check, 14 requests and one episode of 4 steps (about 3 minutes). It prints SMOKE OK or STOP. If it says STOP, do not run the next cells: send me the printout

In [ ]:
a = run_scope_regime(llm, make_scope_tasks(14, 7), True, "/content/smoke_single.jsonl", meta=dict(model=MODEL, smoke=True))
print_scope(a["summary"], "smoke, rule stated")
print("mean seconds per request:", round(mean_seconds(a["rows"]), 2))
b = run_long(llm, "state", 1, 4, 99, True, "/content/smoke_long.jsonl", meta=dict(model=MODEL, smoke=True), policies=("blind", "repair_m6", "scope_m7"), mix="scope")
print_exact(b["summary"], "smoke, 4 steps")
ok, msg = smoke_check(a["summary"])
print("\n" + msg)

### Cell 6: PART 1, rule stated in the prompt. 140 requests, one at a time (about 11 minutes)

In [ ]:
T0 = time.time()
tasks = make_scope_tasks(140, 7)
res_s_rule = run_scope_regime(llm, tasks, True, "/content/out/axi_14b_single_rule.jsonl", meta=dict(model=MODEL, seed=7), progress=prog)
print()
print_scope(res_s_rule["summary"], "PART 1, rule stated, 14B")
print("mean seconds per request:", round(mean_seconds(res_s_rule["rows"]), 2))
files.download(save(res_s_rule, "/content/out/axi_14b_single_rule.json"))

### Cell 7: PART 1, no rule in the prompt. 140 requests (about 6 minutes)

In [ ]:
T0 = time.time()
res_s_norule = run_scope_regime(llm, tasks, False, "/content/out/axi_14b_single_norule.jsonl", meta=dict(model=MODEL, seed=7), progress=prog)
print()
print_scope(res_s_norule["summary"], "PART 1, no rule, 14B")
print("mean seconds per request:", round(mean_seconds(res_s_norule["rows"]), 2))
print_verdict(verdict_big_single({True: res_s_rule["summary"], False: res_s_norule["summary"]}), "PART 1, both regimes")
files.download(save(res_s_norule, "/content/out/axi_14b_single_norule.json"))

### Cell 8: PART 2, rule stated. 6 episodes x 30 requests, 3 policies (about 25 minutes; the counter shows episode i/6)

In [ ]:
T0 = time.time()
res_l_rule = run_long(llm, "state", 6, 30, 7, True, "/content/out/axi_14b_long_rule.jsonl", meta=dict(model=MODEL), policies=("blind", "repair_m6", "scope_m7"), progress=prog, mix="scope")
print()
print_long(res_l_rule["summary"], "PART 2, rule stated, 14B")
print_exact(res_l_rule["summary"], "PART 2, rule stated, 14B")
files.download(save(res_l_rule, "/content/out/axi_14b_long_rule.json"))

### Cell 9: PART 2, no rule. 4 episodes x 30 requests (about 10 minutes)

In [ ]:
T0 = time.time()
res_l_norule = run_long(llm, "state", 4, 30, 7, False, "/content/out/axi_14b_long_norule.jsonl", meta=dict(model=MODEL), policies=("blind", "repair_m6", "scope_m7"), progress=prog, mix="scope")
print()
print_long(res_l_norule["summary"], "PART 2, no rule, 14B")
print_exact(res_l_norule["summary"], "PART 2, no rule, 14B")
print_verdict(verdict_big_long(res_l_rule["summary"], res_l_norule["summary"]), "PART 2, both regimes")
files.download(save(res_l_norule, "/content/out/axi_14b_long_norule.json"))

### Done
Files downloaded: `axi_14b_single_rule.json`, `axi_14b_single_norule.json`, `axi_14b_long_rule.json`, `axi_14b_long_norule.json`. Upload them to the chat, with a screenshot of the printed tables and the 'reading rule' lines. Do not paste any keys anywhere.